In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath("../src"))
from audit_utils import clean_price_to_float, parse_json_safely, audit_modality_coverage

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
df_rev = pd.read_parquet("../data/raw/reviews_Musical_Instruments_sampled.parquet")
df_meta = pd.read_parquet("../data/raw/meta_Musical_Instruments_sampled.parquet")
df_meta["price_numeric"] = df_meta["price_raw"].apply(clean_price_to_float)

print(f"Reviews count: {len(df_rev)}")
print(f"Metadata count: {len(df_meta)}")

In [ ]:
df = pd.merge(df_rev, df_meta, on="parent_asin", how="inner", suffixes=("_rev", "_meta"))
print(f"Merged interactions: {len(df)}")

In [ ]:
modality_summary = audit_modality_coverage(df)
display(modality_summary)

In [ ]:
plt.figure(figsize=(10, 5))
subset_plot = modality_summary.iloc[:9]
bars = plt.barh(subset_plot["modality_aspect"], subset_plot["coverage_percentage"], color="steelblue")
plt.xlabel("Coverage Percentage (%)")
plt.title("Modality Availability Audit - Musical Instruments")
plt.xlim(0, 105)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 1, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va="center")
plt.gca().invert_yaxis()
plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/modality_availability.png", dpi=300)
plt.show()